## Correction notice (2026-08-02)

This notebook originally built the red-noise covariance on a **linearly
spaced** frequency grid. That integral does **not converge** for steep
chromatic spectra: J1909-3744's Band 3 chromatic degradation drifts from
51x at 200 grid points to 18x at 6000, and never settles. The grid below
has been changed to **logarithmic** spacing, which converges by ~400
points and agrees with an independent quadrature rule to 0.5%.

This correction **reversed** the array-level band conclusion, so any
numbers recorded from an earlier run of this notebook are void.

See `scripts/11_grid_convergence.py` and `scripts/12_integrator_crosscheck.py`
for the convergence study. The `scripts/` directory is canonical for all
numbers quoted in the write-up; these notebooks are exploratory.


In [ ]:
# 05_full_array_gwb_sensitivity.ipynb
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import hasasia.sensitivity as hsen
from hasasia.sensitivity import (red_noise_powerlaw, corr_from_psd,
                                 GWBSensitivityCurve)
from hasasia.utils import create_design_matrix

yr = 365.25*24*3600.0
NU_REF, BAND3, BAND5 = 1400.0, 400.0, 1360.0
df = pd.read_csv("inpta_dr2_full_input_table.csv")

# common frequency grid, required: GWB combination needs all spectra aligned
Tmax = df.span_yr.max()*yr
FREQS = np.logspace(np.log10(1/Tmax), np.log10(1e-7), 200)

def chromatic_psd(l10A, g, chi, fr, nu):
    return red_noise_powerlaw(A=10**l10A, freqs=fr, gamma=g)*(nu/NU_REF)**(-2*chi)

def _complete(*vals):
    # True only if every value in a noise-process tuple is finite.
    # A single NaN parameter must not silently enter the covariance matrix --
    # it poisons the whole GWB curve to NaN (caught via J2145-0750 DMN gamma).
    return all(np.isfinite(v) for v in vals)

def build_spec(row, band, cad=24, use_chromatic=True):
    sigma = row.toa_b3_us if band=='b3' else row.toa_b5_us
    nu    = BAND3        if band=='b3' else BAND5
    if not np.isfinite(sigma):
        return None                      # pulsar not observed in this band
    Tspan = row.span_yr*yr
    N_obs = int(round(row.span_yr*cad))
    toas  = np.linspace(0, Tspan, N_obs)
    toaerrs = np.ones(N_obs)*sigma*1e-6
    rn_f = np.logspace(np.log10(1/Tspan), np.log10(N_obs/(2*Tspan)), 800)

    N = np.diag(toaerrs**2)
    if _complete(row.arn_log10A, row.arn_gamma):
        N = N + corr_from_psd(rn_f, red_noise_powerlaw(
                A=10**row.arn_log10A, freqs=rn_f, gamma=row.arn_gamma), toas)
    if use_chromatic and _complete(row.dmn_log10A, row.dmn_gamma):
        N = N + corr_from_psd(rn_f, chromatic_psd(
                row.dmn_log10A, row.dmn_gamma, 2.0, rn_f, nu), toas)
    if use_chromatic and _complete(row.fcn_log10A, row.fcn_gamma, row.fcn_chi):
        N = N + corr_from_psd(rn_f, chromatic_psd(
                row.fcn_log10A, row.fcn_gamma, row.fcn_chi, rn_f, nu), toas)

    M = create_design_matrix(toas, RADEC=True, PROPER=True, PX=True)
    psr = hsen.Pulsar(toas, toaerrs, phi=row.phi, theta=row.theta,
                      designmatrix=M, N=N)
    s = hsen.Spectrum(psr, freqs=FREQS); _ = s.NcalInv
    return s

def array_curve(subset, band, **kw):
    specs = [build_spec(r, band, **kw) for _, r in subset.iterrows()]
    specs = [s for s in specs if s is not None]
    return GWBSensitivityCurve(specs), len(specs)

# --- full array, both bands ---
results = {}
for band in ['b3','b5']:
    gwb, n = array_curve(df, band)
    assert np.isfinite(gwb.h_c).all(), f"NaN in band {band} h_c"
    i = np.argmin(gwb.h_c)
    results[band] = gwb
    print(f"Band {band}: {n} pulsars | h_c,min = {gwb.h_c[i]:.3e} at {gwb.freqs[i]:.2e} Hz")

# --- control: matched subset (pulsars present in BOTH bands) ---
matched = df[df.toa_b3_us.notna() & df.toa_b5_us.notna()]
print(f"\nMatched-subset control ({len(matched)} pulsars):")
for band in ['b3','b5']:
    gwb, n = array_curve(matched, band)
    assert np.isfinite(gwb.h_c).all(), f"NaN in matched-subset band {band} h_c"
    print(f"  Band {band}: h_c,min = {gwb.h_c[np.argmin(gwb.h_c)]:.3e}")

plt.figure(figsize=(7.5,5.5))
plt.loglog(results['b3'].freqs, results['b3'].h_c, label='InPTA DR2 - Band 3 (400 MHz)')
plt.loglog(results['b5'].freqs, results['b5'].h_c, label='InPTA DR2 - Band 5 (1360 MHz)')
plt.xlabel('Frequency [Hz]'); plt.ylabel(r'Characteristic strain $h_c$')
plt.title('InPTA DR2 GWB sensitivity curve (full noise budget)')
plt.legend(); plt.grid(True, which='both', alpha=0.3)
plt.savefig("inpta_dr2_gwb_sensitivity.png", dpi=150, bbox_inches='tight')
plt.show()
